<a href="https://colab.research.google.com/github/heytian/d2d-oco3-tools/blob/main/20260404_02_updateDuckDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Update DuckDB (OCO-3 CO2 & SIF) based on Clasp Report 356 SAMs**


- 20260404 Step 2 to "20260404_01_centroids+ne.py" Step 1 in same git repo.
- Background: Step 1 was to update Clasp Report (centroid location information from JPL team on SAM targets) with Natural Earth population and city name information
- **IMPORTANT**: Clear all outputs and end runtime session before saving to Colab or Github to avoid exposing your Earthdata credentials!


**Development notes (Work in progress as of Apr 4, 2026): this is with reference to the NetCDF to DuckDB pipeline from March 2026 named "nc4-timezone-pop.ipynb" in the same github repo.



In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import duckdb
import geopandas as gpd
import os, shutil
import pandas as pd
from shapely import wkt

DB_SRC = "/content/drive/MyDrive/Shortcuts/columnar/oco3_data.duckdb"
con = duckdb.connect(DB_SRC, read_only=True)


In [4]:
# Data exploration for DuckDB
# run 1 "con.execute" command at a time

con.execute("SHOW TABLES").df()

con.execute("SELECT * FROM co2_sam LIMIT 5").df()

con.execute("""
    SELECT COUNT(*)                    AS rows,
           COUNT(DISTINCT source_file) AS files,
           COUNT(DISTINCT target_name) AS targets,
           MIN(datetime)               AS earliest,
           MAX(datetime)               AS latest
    FROM co2_sam
""").fetchone()

con.execute("DESCRIBE co2_sam").df()

,column_name,column_type,null,key,default,extra
0,xco2,FLOAT,YES,None,None,None
1,source_file,VARCHAR,YES,None,None,None
2,target_name,VARCHAR,YES,None,None,None
3,datetime,TIMESTAMP_NS,YES,None,None,None
4,local_time,TIMESTAMP_NS,YES,None,None,None
5,latitude,FLOAT,YES,None,None,None
6,longitude,FLOAT,YES,None,None,None
7,city,VARCHAR,YES,None,None,None
8,country,VARCHAR,YES,None,None,None
9,population,BIGINT,YES,None,None,None


In [3]:
# !pip install duckdb geopandas shapely --quiet

DB_SRC    = "/content/drive/MyDrive/Shortcuts/columnar/oco3_data.duckdb"
DB_LOCAL  = "/content/oco3_data.duckdb" # add local path for stability when processing DuckDB file
WKT_PATH  = "/content/drive/MyDrive/Shortcuts/csv_xlsx/20260129_from_Rob/clasp_report_356sams.csv" #updated clasp report with NE population and cities added

PARQUET_DIR = "/content/drive/MyDrive/Shortcuts/columnar/oco3_parquet"
CSV_DIR     = "/content/drive/MyDrive/Shortcuts/csv_xlsx/from-duckdb"

TABLES = {
    "co2_sam": "xco2",
    "sif_sam":  "Daily_SIF_757nm",
}

print("Loading updated clasp_report...")
ref_data    = pd.read_csv(WKT_PATH)
ref_geodata = gpd.GeoDataFrame(
    ref_data,
    geometry=ref_data["Site Shape WKT"].apply(wkt.loads),
    crs="EPSG:4326"
)

SAM_city_pop = ref_data.set_index("Target Name")[["city", "country", "population"]]

print(f"  Information from {len(ref_geodata)} SAM locations loaded")
print(f"  Sample:\n{SAM_city_pop.head(3)}")

def spatial_filter(df):

    df = df.drop(columns=['city','country','population'],errors='ignore')

    pts = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    )
    joined = gpd.sjoin(
        pts,
        ref_geodata[["Target Name", "geometry"]],
        how="inner", predicate="within"
    )
    if joined.empty:
        return None

    joined = joined.rename(columns={"Target Name": "target_name_new"})
    joined = joined.drop(columns=["geometry", "index_right"], errors = "ignore")

    original_match = joined["target_name_new"] == joined["target_name"]
    keep           = joined[original_match].copy()

    resolved_key   = keep["latitude"].astype(str) + "_" + keep["longitude"].astype(str)
    unresolved     = joined[~(joined["latitude"].astype(str) + "_" + joined["longitude"].astype(str)).isin(resolved_key)]
    fallback       = unresolved.drop_duplicates(subset=["latitude", "longitude"])

    # debugging to see which does not have matching target name between OCO3 and clasp report
    print(f"    Matched by existing target_name: {len(keep):,}")
    print(f"    Fallback (first match):          {len(fallback):,}")

    joined = pd.concat([keep, fallback], ignore_index=True)
    joined = joined.drop(columns=["target_name"])
    joined = joined.rename(columns={"target_name_new": "target_name"})

    joined = joined.join(SAM_city_pop, on="target_name", how="left")
    return joined

print("\nCopying DB locally...")
shutil.copy(DB_SRC, DB_LOCAL)
print(f"  Local DB: {os.path.getsize(DB_LOCAL)/1024**2:.1f} MB")

con = duckdb.connect(DB_LOCAL)

for table in TABLES:
    print(f"Processing: {table}")

    try:
        total_before = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  Rows before: {total_before:,}")
    except Exception as e:
        print(f"  Table {table} not found: {e}, skipping")
        continue

    df = con.execute(f"SELECT * FROM {table}").df()
    print(f"  Loaded {len(df):,} rows")

    print("  Running spatial filter...")
    filtered = spatial_filter(df)

    if filtered is None or filtered.empty:
        print(f"  WARNING: No rows survived spatial filter for {table}!")
        continue

    print(f"  Rows after filter: {len(filtered):,} (dropped {len(df)-len(filtered):,})")

    original_cols = list(con.execute(f"DESCRIBE {table}").df()['column_name'])
    final_cols    = [c for c in original_cols if c in filtered.columns]

    filtered = filtered[final_cols]

    print(f"  Truncating and reloading {table}...")
    con.execute(f"DELETE FROM {table}")
    con.execute(f"INSERT INTO {table} SELECT * FROM filtered")

    total_after = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  Rows after reload: {total_after:,}")

con.close()

print("\n── Verification ──────────────────────────")
con = duckdb.connect(DB_LOCAL)
for table in TABLES:
    try:
        row = con.execute(f"""
            SELECT COUNT(*)                    AS rows,
                   COUNT(DISTINCT target_name) AS targets,
                   COUNT(DISTINCT city)        AS cities,
                   MIN(datetime)               AS earliest,
                   MAX(datetime)               AS latest
            FROM {table}
        """).fetchone()
        print(f"{table}: {row[0]:,} rows | {row[1]} targets | {row[2]} cities | {row[3]} → {row[4]}")
    except Exception as e:
        print(f"{table}: {e}")
con.close()

print("\nCopying back to Drive...")
shutil.copy(DB_LOCAL, DB_SRC)
print(f"  Drive DB: {os.path.getsize(DB_SRC)/1024**2:.1f} MB")

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

con = duckdb.connect(DB_SRC)
for table in TABLES:
    parquet_path = f"{PARQUET_DIR}/{table}.parquet"
    csv_path     = f"{CSV_DIR}/{table}.csv"

    con.execute(f"COPY (SELECT * FROM {table}) TO '{parquet_path}' (FORMAT PARQUET)")
    print(f"  Parquet: {table} updated, file size: {os.path.getsize(parquet_path)/1024**2:.1f} MB")

    con.execute(f"COPY {table} TO '{csv_path}' (HEADER, DELIMITER ',')")
    print(f"  CSV:     {table} updated, file size: {os.path.getsize(csv_path)/1024**2:.1f} MB")

con.close()
print("\nFin.")

Loading updated clasp_report...
  Information from 356 SAM locations loaded
  Sample:
                               city      country  population
Target Name                                                 
Auckland_NewZealand        Auckland  New Zealand     1377200
Dakar_Senegal                 Thies      Senegal      293001
DaresSalaam_Tanzania  Dar es Salaam     Tanzania     2930000

Copying DB locally...
  Local DB: 589.3 MB
Processing: co2_sam
  Rows before: 6,053,902
  Loaded 6,053,902 rows
  Running spatial filter...
    Matched by existing target_name: 4,815,474
    Fallback (first match):          0
  Rows after filter: 4,815,474 (dropped 1,238,428)
  Truncating and reloading co2_sam...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Rows after reload: 4,815,474
Processing: sif_sam
  Rows before: 18,929,840


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Loaded 18,929,840 rows
  Running spatial filter...
    Matched by existing target_name: 13,666,431
    Fallback (first match):          0
  Rows after filter: 13,666,431 (dropped 5,263,409)
  Truncating and reloading sif_sam...


...

  Rows after reload: 13,666,431

── Verification ──────────────────────────
co2_sam: 4,815,474 rows | 345 targets | 320 cities | 2019-08-06 08:53:08 → 2025-12-29 02:09:33
sif_sam: 13,666,431 rows | 346 targets | 321 cities | 2019-08-06 04:00:17.617187 → 2025-12-29 02:11:28.726562

Copying back to Drive...
  Drive DB: 426.8 MB
  Parquet: co2_sam → 65.2 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  CSV:     co2_sam → 761.7 MB
  Parquet: sif_sam → 334.1 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  CSV:     sif_sam → 2352.5 MB

All done.
